In [1]:
from pathlib import Path
import shutil

import geopandas as gpd
import pandas as pd

from satchip import models, download_data, merge_modality, generate_labels, chip_data, view

/home/wbhorn/miniforge3/envs/satchip/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
shp_path = 'Pristine_Merged'
df = gpd.read_file(shp_path)

df['SwathDate'] = pd.to_datetime(df['SwathDate'], format='%Y-%m-%d')
df['HLSDate'] = pd.to_datetime(df['HLSDate'], format='%Y-%m-%d')

#df = df[df['Visibility'] == '1']

In [3]:
df


,SwathDate,HLSID,MODISID,Visibility,HLSDate,Field,geometry
0,2019-07-09,102a,102,1,2019-07-17,S2,"POLYGON Z ((-99.49952 40.26693 0, -99.47639 40..."
1,2019-08-13,106a,106,2,2019-08-28,LS,"POLYGON Z ((-101.59283 40.78718 0, -101.60341 ..."
2,2018-08-06,126a,126,1,2018-08-11,S2,"POLYGON Z ((-98.2416 41.16581 0, -98.24082 41...."
3,2018-08-06,127a,127,1,2018-08-11,S2,"POLYGON Z ((-97.78549 41.00092 0, -97.76367 41..."
4,2018-07-29,129a,129,3,2018-08-07,LS,"POLYGON Z ((-103.1513 43.15378 0, -103.14676 4..."
5,2018-07-27,130a,130,2,2018-08-07,LS,"POLYGON Z ((-102.98845 43.56544 0, -102.97929 ..."
6,2018-07-27,132a,132,3,2018-08-11,LS,"MULTIPOLYGON Z (((-97.42377 41.60069 0, -97.42..."
7,2018-07-18,133a,133,2,2018-07-26,LS,"POLYGON Z ((-97.5217 42.65295 0, -97.5154 42.6..."
8,2018-06-30,134a,134,1,2018-07-12,S2,"POLYGON Z ((-99.72435 40.41203 0, -99.70477 40..."
9,2017-07-05,598a,598,2,2017-07-15,S2,"POLYGON Z ((-98.60738 45.24939 0, -98.60167 45..."


In [4]:
DATA_PATH = Path.cwd() / 'data'

def make_mod_paths(modality):
    base_path = DATA_PATH / modality['id']

    return {
        'modality': modality,
        'raw': base_path / 'raw',
        'merged': base_path / 'merged',
        'wgs84': base_path / 'wgs84',
        'stacked': base_path / 'stacked',
        'warped': base_path / 'warped',
        'chips': base_path / 'chips',
        'plots': base_path / 'plots'
    }

mod_paths = {
    modality['id']: make_mod_paths(modality) for modality in (models.HLS_S30, models.HLS_L30)
}
mod_paths

{'HLS_S30': {'modality': {'id': 'HLS_S30',
   'collection': 'HLSS30',
   'bands': (Band(id='B02', name='Blue', shortname='B'),
    Band(id='B03', name='Green', shortname='G'),
    Band(id='B04', name='Red', shortname='R'),
    Band(id='B8A', name='NIR Narrow', shortname='N'),
    Band(id='B11', name='SWIR 1', shortname='SW1'),
    Band(id='B12', name='SWIR 2', shortname='SW2'),
    Band(id='Fmask', name='Cloud Mask', shortname='Fmask'))},
  'raw': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/raw'),
  'merged': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/merged'),
  'wgs84': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/wgs84'),
  'stacked': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/stacked'),
  'warped': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/warped'),
  'chips': PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/chips'),
  'plot

In [5]:
event_files = []

for paths in mod_paths.values():
    print(paths['modality'], paths['raw'])

    for idx, row in df.iterrows():
        modality = paths['modality']
        best_mod =  models.HLS_L30 if row['Field'] == 'LS' else models.HLS_S30
        if modality != best_mod:
            continue

        damage_event = models.Event(
            name=row['HLSID'], 
            date=row['HLSDate'], 
            wgs84_geometry=row['geometry'], 
            buffer_m=10000
        )

        local = download_data.download_data(damage_event, modality, paths['raw'])

        if not local:
            print(f'no {modality["id"]} data for {damage_event.name}')
            continue
        else:
            print(f'Found {modality["id"]} data for {damage_event.name}')

        merged_event = merge_modality.merge_modality(
            local, 
            modality, 
            event=damage_event, 
            output_path=paths['merged']
        )

        stacked_filename = paths['stacked'] / f'{damage_event.name}.{modality["id"]}.stacked.tif'
        data_bands, fmask = merged_event[:-1], merged_event[-1]

        stacked = merge_modality.stack_bands(data_bands, stacked_filename)

        label = generate_labels.binary_mask_from_template(stacked, damage_event, paths['wgs84'])
        mask, bands, fmask = merge_modality.warp_to_reference(
            reference_path=label,
            data_files=[stacked, fmask],
            output_dir=paths['warped'],
            bounding_box_wgs84=damage_event.buffered_geometry().bounds,
        )
        print(f'Adding event files for {damage_event.name}')
        event_files.append((damage_event, modality, (mask, bands, fmask)))

        view.view_merged(
            bands, 
            damage_event, 
            modality, 
            rgb_bands=[2, 1, 0],
            quite=True, 
            save_to_file=paths['plots'] / f'{damage_event.name}.{modality["id"]}.merged.plot.png'
        )

{'id': 'HLS_S30', 'collection': 'HLSS30', 'bands': (Band(id='B02', name='Blue', shortname='B'), Band(id='B03', name='Green', shortname='G'), Band(id='B04', name='Red', shortname='R'), Band(id='B8A', name='NIR Narrow', shortname='N'), Band(id='B11', name='SWIR 1', shortname='SW1'), Band(id='B12', name='SWIR 2', shortname='SW2'), Band(id='Fmask', name='Cloud Mask', shortname='Fmask'))} /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/raw
Logging in to earthaccess


QUEUEING TASKS | : 100%|██████████| 36/36 [00:00<00:00, 13296.49it/s]
PROCESSING TASKS | : 100%|██████████| 36/36 [00:00<00:00, 371908.73it/s]
COLLECTING RESULTS | : 100%|██████████| 36/36 [00:00<00:00, 725937.23it/s]


Found HLS_S30 data for 102a
generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/wgs84/102a.MASK.tif
Adding event files for 102a


QUEUEING TASKS | : 100%|██████████| 18/18 [00:00<00:00, 11018.31it/s]
PROCESSING TASKS | : 100%|██████████| 18/18 [00:00<00:00, 324023.48it/s]
COLLECTING RESULTS | : 100%|██████████| 18/18 [00:00<00:00, 431414.13it/s]


Found HLS_S30 data for 126a
generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/wgs84/126a.MASK.tif
Adding event files for 126a


...

Found HLS_L30 data for 1378a
generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/wgs84/1378a.MASK.tif
Adding event files for 1378a


QUEUEING TASKS | : 100%|██████████| 30/30 [00:00<00:00, 13177.20it/s]
PROCESSING TASKS | : 100%|██████████| 30/30 [00:00<00:00, 418036.94it/s]
COLLECTING RESULTS | : 100%|██████████| 30/30 [00:00<00:00, 748982.86it/s]


Found HLS_L30 data for 1052a
generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/wgs84/1052a.MASK.tif
Adding event files for 1052a


QUEUEING TASKS | : 100%|██████████| 15/15 [00:00<00:00, 11670.29it/s]
PROCESSING TASKS | : 100%|██████████| 15/15 [00:00<00:00, 322638.77it/s]
COLLECTING RESULTS | : 100%|██████████| 15/15 [00:00<00:00, 491520.00it/s]


Found HLS_L30 data for 1376a
generated: /home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/wgs84/1376a.MASK.tif
Adding event files for 1376a


Remove any bad aquisitions from `data/{modality}/plots` by remove the .png 

In [7]:
def backup_plots():
    for paths in mod_paths.values():
        plots = paths['plots']
        plots_backup = plots.parent / 'plots-backup'
        shutil.copytree(plots, plots_backup, dirs_exist_ok=True)

# backup_plots()

In [8]:
def restore_plots():
    for paths in mod_paths.values():
        plots = paths['plots']
        plots_backup = plots.parent / 'plots-backup'
        shutil.copytree(plots_backup, plots, dirs_exist_ok=True)

# restore_plots()

In [9]:
keepers = {}

for paths in mod_paths.values():
    modality = paths['modality']
    mod_keepers = [plot.name.split('.')[0] for plot in paths['plots'].glob('*.png')]
    keepers[modality['id']] = mod_keepers

keepers

{'HLS_S30': ['628a',
  '1079a',
  '1069a',
  '116d',
  '648a',
  '638a',
  '116e',
  '1338a',
  '598a',
  '1347a',
  '102a',
  '110d',
  '1373e',
  '892a',
  '1373d',
  '1064a',
  '1056a',
  '889b',
  '1055a',
  '889a',
  '126a',
  '134a',
  '127a',
  '1373a'],
 'HLS_L30': ['132a',
  '1378a',
  '1376a',
  '133a',
  '915a',
  '1052a',
  '623d',
  '130a',
  '614a',
  '106a',
  '623a',
  '129a']}

In [10]:
print(event_files[0])
evt, modality, (m, b, f) = event_files[0]

(Event(name='102a', date=Timestamp('2019-07-17 00:00:00'), wgs84_geometry=<POLYGON Z ((-99.5 40.267 0, -99.476 40.268 0, -99.458 40.275 0, -99.43 40.2...>, buffer_m=10000), {'id': 'HLS_S30', 'collection': 'HLSS30', 'bands': (Band(id='B02', name='Blue', shortname='B'), Band(id='B03', name='Green', shortname='G'), Band(id='B04', name='Red', shortname='R'), Band(id='B8A', name='NIR Narrow', shortname='N'), Band(id='B11', name='SWIR 1', shortname='SW1'), Band(id='B12', name='SWIR 2', shortname='SW2'), Band(id='Fmask', name='Cloud Mask', shortname='Fmask'))}, (PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/warped/102a.MASK.tif'), PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/warped/102a.HLS_S30.stacked.tif'), PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_S30/warped/102a.HLS_S30.2019-07-17.Fmask.tif')))


In [11]:
all_base = DATA_PATH / 'chips'
output_base = DATA_PATH / 'output'

chip_paths = {
    'all': {
        'label': all_base / 'LABEL',
        'hls': all_base / 'HLS',
        'other': all_base / 'OTHER',
        'plots': all_base / 'PLOTS',
    },
    'output': {
        'label': output_base / 'LABEL',
        'hls': output_base / 'HLS',
        'other': output_base / 'OTHER',
        'plots': all_base / 'PLOTS',
    }
}

In [12]:
chipped_events = set() 
event_chip_stacks = []

for evt, modality, (m, b, f) in event_files:
    if evt.name not in keepers[modality['id']]:
        continue

    print('Chipping: ', evt.name, m.name, b.name, f.name)
    if evt.name in chipped_events:
        print(f'Already chipped event {evt.name}: skipping...')
        continue

    chipped_events.add(evt.name)
    grid = chip_data.make_grid_from_reference(m)

    label_chips = chip_data.chip_data(grid, m, chip_paths['all']['label'])
    data_chips = chip_data.chip_data(grid, b, chip_paths['all']['hls'])
    fmask_chips = chip_data.chip_data(grid, f, chip_paths['all']['other'])

    event_chips = chip_data.make_chip_stacks(data_chips, fmask_chips, label_chips, modality)
    event_chip_stacks.append((event_chips, evt))

Chipping:  102a 102a.MASK.tif 102a.HLS_S30.stacked.tif 102a.HLS_S30.2019-07-17.Fmask.tif
Chipping:  126a 126a.MASK.tif 126a.HLS_S30.stacked.tif 126a.HLS_S30.2018-08-11.Fmask.tif
Chipping:  127a 127a.MASK.tif 127a.HLS_S30.stacked.tif 127a.HLS_S30.2018-08-11.Fmask.tif
Chipping:  134a 134a.MASK.tif 134a.HLS_S30.stacked.tif 134a.HLS_S30.2018-07-12.Fmask.tif
Chipping:  598a 598a.MASK.tif 598a.HLS_S30.stacked.tif 598a.HLS_S30.2017-07-15.Fmask.tif
Chipping:  889a 889a.MASK.tif 889a.HLS_S30.stacked.tif 889a.HLS_S30.2018-07-13.Fmask.tif
Chipping:  889b 889b.MASK.tif 889b.HLS_S30.stacked.tif 889b.HLS_S30.2018-07-13.Fmask.tif
Chipping:  892a 892a.MASK.tif 892a.HLS_S30.stacked.tif 892a.HLS_S30.2018-07-08.Fmask.tif
Chipping:  1079a 1079a.MASK.tif 1079a.HLS_S30.stacked.tif 1079a.HLS_S30.2019-08-04.Fmask.tif
Chipping:  1069a 1069a.MASK.tif 1069a.HLS_S30.stacked.tif 1069a.HLS_S30.2019-08-14.Fmask.tif
Chipping:  628a 628a.MASK.tif 628a.HLS_S30.stacked.tif 628a.HLS_S30.2019-07-13.Fmask.tif
Chipping:  63

In [13]:
print(event_chip_stacks[0])

([ChipStack(id='000.000', data=PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/chips/HLS/000.000.102a.HLS_S30.stacked.tif'), validation_mask=PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/chips/OTHER/000.000.102a.HLS_S30.2019-07-17.Fmask.tif'), label=PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/chips/LABEL/000.000.102a.MASK.tif'), modality={'id': 'HLS_S30', 'collection': 'HLSS30', 'bands': (Band(id='B02', name='Blue', shortname='B'), Band(id='B03', name='Green', shortname='G'), Band(id='B04', name='Red', shortname='R'), Band(id='B8A', name='NIR Narrow', shortname='N'), Band(id='B11', name='SWIR 1', shortname='SW1'), Band(id='B12', name='SWIR 2', shortname='SW2'), Band(id='Fmask', name='Cloud Mask', shortname='Fmask'))}), ChipStack(id='000.001', data=PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/chips/HLS/000.001.102a.HLS_S30.stacked.tif'), validation_mask=PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/chi

In [19]:
import random

all_chips = []
filtered_chips = []
all_no_damage = []
all_damage = []

for event_chips, event in event_chip_stacks:
    print(f'filtering chips for {event.name}')
    all_chips += event_chips
    view.view_chips(
        event_chips, 
        modality, 
        rgb_bands=[2, 1, 0], 
        save_to_file=chip_paths['all']['plots'] / f'{event.name}.chips.png', 
        quite=True
    )
    
    damage_chips = chip_data.filter_damage_chips(event_chips)
    all_damage += damage_chips

    no_damage_chips = chip_data.filter_no_damage_chips(event_chips)

    random.shuffle(no_damage_chips)
    no_damage_chips = no_damage_chips[: int(len(damage_chips) * 1.5)]

    print(f'  damage: {len(damage_chips)}, no damage: {len(no_damage_chips)}')

    if len(damage_chips) == 0 and len(no_damage_chips) == 0:
        print(f'Filtered out all chips for {event.name}')
        continue

    all_no_damage += no_damage_chips
    filtered = damage_chips + no_damage_chips
    
    view.view_chips(
        filtered, 
        modality, 
        rgb_bands=[2, 1, 0], 
        save_to_file=chip_paths['output']['plots'] / f'{event.name}.filtered.png', 
        quite=True
    )
    
    filtered_chips += filtered

filtering chips for 102a
  damage: 10, no damage: 15
filtering chips for 126a
  damage: 3, no damage: 4
filtering chips for 127a
  damage: 16, no damage: 24
filtering chips for 134a
  damage: 8, no damage: 12
filtering chips for 598a
  damage: 7, no damage: 10
filtering chips for 889a
  damage: 10, no damage: 15
filtering chips for 889b
  damage: 5, no damage: 7
filtering chips for 892a
  damage: 15, no damage: 22
filtering chips for 1079a
  damage: 25, no damage: 37
filtering chips for 1069a
  damage: 3, no damage: 4
filtering chips for 628a
  damage: 1, no damage: 1
filtering chips for 638a
  damage: 47, no damage: 70
filtering chips for 116d
  damage: 6, no damage: 9
filtering chips for 116e
  damage: 3, no damage: 4
filtering chips for 648a
  damage: 4, no damage: 6
filtering chips for 1055a
  damage: 24, no damage: 36
filtering chips for 1056a
  damage: 22, no damage: 33
filtering chips for 1347a
  damage: 18, no damage: 27
filtering chips for 1064a
  damage: 13, no damage: 19
fil

In [21]:
len(all_damage), len(all_no_damage), len(filtered_chips), len(all_chips)

(426, 623, 1049, 1688)

In [22]:
chip_paths['output']['hls']

PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/output/HLS')

In [23]:
chip_paths['output']['hls'].mkdir(exist_ok=True, parents=True)
chip_paths['output']['label'].mkdir(exist_ok=True, parents=True)

for chip in filtered_chips:
    output_data_path = chip_paths['output']['hls'] / chip.data.name 
    output_label_path = chip_paths['output']['label'] / chip.label.name

    shutil.copy2(chip.data, output_data_path)
    shutil.copy2(chip.label, output_label_path)

In [24]:
len(list(chip_paths['output']['hls'].glob('*.tif')))
len(list(chip_paths['output']['label'].glob('*.tif')))

1049

In [25]:
#len(filtered_chips)
#filtered_chips[0]
#CHIPS_PLOTS = PLOT_PATH / 'CHIPS'
#print(len(filtered_chips), len(all_chips))
#for chip in filtered_chips:
#    plot_name = f"{chip.data.name.split('.stacked')[0]}.chip.png"
#    view.view_chip(chip, models.HLS_S30, [2, 1, 0], save_to_file=CHIPS_PLOTS / plot_name, quite=True)

In [26]:
import numpy as np
import rasterio


def calculate_stats(chip_stacks, n_bands) -> tuple:
    mean = np.zeros(n_bands, dtype=np.float64)
    M2 = np.zeros(n_bands, dtype=np.float64)
    count = np.zeros(n_bands, dtype=np.float64)

    for chip in chip_stacks:
        with rasterio.open(chip.data) as src:
            band_data = src.read()
            count, mean, M2 = 0, 0, 0

            _, H, W = band_data.shape

            batch_count = H * W
            batch_mean = band_data.mean(axis=(1, 2))
            batch_var = band_data.var(axis=(1, 2))

            delta = batch_mean - mean
            total_count = count + batch_count

            mean = mean + delta * (batch_count / total_count)
            M2 = (
                M2
                + batch_var * batch_count
                + (delta**2) * count * batch_count / total_count
            )
            count = total_count

    variance = M2 / count
    std = np.sqrt(variance)

    return mean, std


In [27]:
data_bands

[PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.B.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.G.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.R.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.N.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.SW1.tif'),
 PosixPath('/home/wbhorn/Repositories/fm/satchip/notebooks/data/HLS_L30/merged/1376a.HLS_L30.2020-07-19.SW2.tif')]

In [28]:
chip_means, chip_stds = calculate_stats(filtered_chips, 6)

In [29]:
import json


bands = models.HLS_S30['bands'][:-1]
bands

stats_file = {
    'HLS':{
        'means': {},
        'stds': {},
    }
}

for mean, std, band in zip(chip_means, chip_stds, bands):
    stats_file['HLS']['means'][band.shortname] = float(mean)
    stats_file['HLS']['stds'][band.shortname] = float(std)

stats_file

(output_base / 'statistics.json').write_text(json.dumps(stats_file, indent=2))
stats_file

{'HLS': {'means': {'B': 267.3031005859375,
   'G': 500.4424133300781,
   'R': 332.49700927734375,
   'N': 4312.5751953125,
   'SW1': 1867.71044921875,
   'SW2': 887.797607421875},
  'stds': {'B': 112.45736694335938,
   'G': 160.5001678466797,
   'R': 199.83193969726562,
   'N': 594.2197265625,
   'SW1': 496.0038757324219,
   'SW2': 382.4466857910156}}}